# Speech model performance

This notebook builds the ADReSSo segmentation feature tables, then creates model-focused figures here in the notebook. If labeled `y_true`/`y_pred` results exist, it plots real performance metrics; otherwise it audits prediction-output files for completion and validity instead of creating dataset-only figures.

In [ ]:
from pathlib import Path
import json
import os
import sys
from typing import Any, Sequence

NOTEBOOK_CWD = Path.cwd()
if (NOTEBOOK_CWD / "preprocessing" / "speech" / "analyze_cha_stats.py").exists():
    REPO_ROOT = NOTEBOOK_CWD
else:
    REPO_ROOT = Path("../..").resolve()

CACHE_DIR = REPO_ROOT / "output_performance" / "speech" / ".cache"
os.environ.setdefault("MPLCONFIGDIR", str(CACHE_DIR / "matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(CACHE_DIR / "xdg"))

repo_path = str(REPO_ROOT.resolve())
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

if "ipykernel" not in sys.modules:
    import matplotlib
    matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    precision_recall_curve,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

from preprocessing.speech.analyze_cha_stats import run_analysis

PERFORMANCE_COLUMN_HINT = (
    "Expected a CSV with labeled model outputs. Classification columns: "
    "ID optional, y_true, y_pred, and optional y_score/probability. "
    "Regression columns: ID optional, y_true and y_pred."
)
PREDICTION_OUTPUT_HINT = (
    "Unlabeled prediction-output files can still produce model-output quality "
    "figures when they include ID and Prediction/y_pred columns."
)
PREDICTION_OUTPUT_GLOBS = (
    "test_results_task*.csv",
    "test_results-task*.csv",
    "model_predictions*.csv",
    "model_outputs*.csv",
    "predictions*.csv",
)

class ModelPerformanceDataError(ValueError):
    pass

print(f"Using speech directory: {SPEECH_DIR.resolve()}")

In [ ]:
def _find_column(df: pd.DataFrame, candidates: tuple[str, ...]) -> str | None:
    normalized = {str(col).strip().lower(): col for col in df.columns}
    for candidate in candidates:
        if candidate.lower() in normalized:
            return normalized[candidate.lower()]
    return None


def _numeric_if_possible(series: pd.Series) -> pd.Series:
    numeric = pd.to_numeric(series, errors="coerce")
    if numeric.notna().sum() == series.notna().sum():
        return numeric
    return series.astype(str)


def _json_ready(value: Any) -> Any:
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, dict):
        return {str(key): _json_ready(val) for key, val in value.items()}
    if isinstance(value, list):
        return [_json_ready(item) for item in value]
    return value


def prepare_model_results(results: pd.DataFrame, task: str = "auto") -> tuple[pd.DataFrame, str]:
    y_true_col = _find_column(
        results,
        ("y_true", "true_label", "label", "target", "actual", "truth", "true_mmse", "mmse"),
    )
    y_pred_col = _find_column(
        results,
        ("y_pred", "prediction", "pred", "predicted", "predicted_label", "predicted_mmse"),
    )
    score_col = _find_column(
        results,
        (
            "y_score",
            "score",
            "probability",
            "prob",
            "prob_ad",
            "ad_probability",
            "positive_probability",
            "p_ad",
        ),
    )
    id_col = _find_column(results, ("id", "subject_id", "file", "filename"))

    if y_true_col is None or y_pred_col is None:
        raise ModelPerformanceDataError(PERFORMANCE_COLUMN_HINT)

    prepared = pd.DataFrame({"y_true": results[y_true_col], "y_pred": results[y_pred_col]})
    if id_col is not None:
        prepared.insert(0, "ID", results[id_col])
    if score_col is not None and score_col != y_pred_col:
        prepared["y_score"] = pd.to_numeric(results[score_col], errors="coerce")

    prepared = prepared.dropna(subset=["y_true", "y_pred"]).reset_index(drop=True)
    if prepared.empty:
        raise ModelPerformanceDataError("Model results are empty after dropping missing labels/predictions.")

    prepared["y_true"] = _numeric_if_possible(prepared["y_true"])
    prepared["y_pred"] = _numeric_if_possible(prepared["y_pred"])
    true_unique = prepared["y_true"].dropna().unique()
    pred_unique = prepared["y_pred"].dropna().unique()
    pred_numeric = pd.to_numeric(prepared["y_pred"], errors="coerce")
    pred_is_probability = (
        pred_numeric.notna().sum() == prepared["y_pred"].notna().sum()
        and pred_numeric.between(0, 1).all()
        and not set(pred_numeric.dropna().unique()).issubset({0, 1})
    )

    if task == "auto":
        task = (
            "classification"
            if len(true_unique) <= 2 and (len(pred_unique) <= 2 or pred_is_probability or "y_score" in prepared.columns)
            else "regression"
        )
    if task not in {"classification", "regression"}:
        raise ValueError("task must be 'auto', 'classification', or 'regression'")

    if task == "classification":
        if "y_score" not in prepared.columns and pd.api.types.is_numeric_dtype(prepared["y_pred"]):
            pred_values = prepared["y_pred"].astype(float)
            if pred_values.between(0, 1).all() and not set(pred_values.dropna().unique()).issubset({0, 1}):
                prepared["y_score"] = pred_values
                prepared["y_pred"] = (pred_values >= 0.5).astype(int)
    else:
        prepared["y_true"] = pd.to_numeric(prepared["y_true"], errors="coerce")
        prepared["y_pred"] = pd.to_numeric(prepared["y_pred"], errors="coerce")
        prepared = prepared.dropna(subset=["y_true", "y_pred"]).reset_index(drop=True)
        if prepared.empty:
            raise ModelPerformanceDataError("Regression performance requires numeric y_true and y_pred values.")

    return prepared, task


def compute_classification_metrics(results: pd.DataFrame) -> dict[str, Any]:
    y_true = results["y_true"]
    y_pred = results["y_pred"]
    labels = sorted(pd.unique(pd.concat([y_true, y_pred], ignore_index=True)))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    metrics: dict[str, Any] = {
        "task": "classification",
        "n": int(len(results)),
        "labels": [str(label) for label in labels],
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "confusion_matrix": cm,
    }

    if len(labels) == 2:
        positive_label = labels[-1]
        y_true_bin = (y_true == positive_label).astype(int)
        y_pred_bin = (y_pred == positive_label).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true_bin, y_pred_bin, labels=[0, 1]).ravel()
        metrics.update(
            {
                "positive_label": str(positive_label),
                "sensitivity": float(recall_score(y_true_bin, y_pred_bin, zero_division=0)),
                "specificity": float(tn / (tn + fp)) if (tn + fp) else 0.0,
                "precision": float(precision_score(y_true_bin, y_pred_bin, zero_division=0)),
                "f1": float(f1_score(y_true_bin, y_pred_bin, zero_division=0)),
            }
        )
        if "y_score" in results.columns and results["y_score"].notna().all():
            y_score = results["y_score"].astype(float)
            metrics["roc_auc"] = float(roc_auc_score(y_true_bin, y_score))
            metrics["average_precision"] = float(average_precision_score(y_true_bin, y_score))

    return metrics


def compute_regression_metrics(results: pd.DataFrame) -> dict[str, Any]:
    y_true = results["y_true"].astype(float)
    y_pred = results["y_pred"].astype(float)
    residuals = y_pred - y_true
    rmse = float(np.sqrt(np.mean(np.square(residuals))))
    return {
        "task": "regression",
        "n": int(len(results)),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": rmse,
        "r2": float(r2_score(y_true, y_pred)),
        "mean_error": float(residuals.mean()),
        "median_absolute_error": float(np.median(np.abs(residuals))),
    }

In [ ]:
def plot_classification_performance(results: pd.DataFrame, metrics: dict[str, Any]) -> plt.Figure:
    plt.style.use("seaborn-v0_8-whitegrid")
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    fig.suptitle("Speech Model Classification Performance", fontsize=15)

    cm = np.array(metrics["confusion_matrix"])
    labels = metrics["labels"]
    ax = axes[0, 0]
    im = ax.imshow(cm, cmap="Blues")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title("Confusion Matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticks(range(len(labels)), labels=labels)
    ax.set_yticks(range(len(labels)), labels=labels)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", color="#111827")

    metric_keys = ["accuracy", "sensitivity", "specificity", "precision", "f1", "roc_auc"]
    available = [(key, metrics[key]) for key in metric_keys if key in metrics]
    axes[0, 1].bar([key.replace("_", " ").title() for key, _ in available], [v for _, v in available], color="#2563eb")
    axes[0, 1].set_ylim(0, 1)
    axes[0, 1].set_title("Core Metrics")
    axes[0, 1].tick_params(axis="x", labelrotation=35)

    if "y_score" in results.columns and results["y_score"].notna().all() and len(labels) == 2:
        labels_raw = sorted(pd.unique(pd.concat([results["y_true"], results["y_pred"]], ignore_index=True)))
        positive_label = labels_raw[-1]
        y_true_bin = (results["y_true"] == positive_label).astype(int)
        y_score = results["y_score"].astype(float)

        fpr, tpr, _ = roc_curve(y_true_bin, y_score)
        axes[1, 0].plot(fpr, tpr, color="#2563eb", lw=2.2, label=f"AUC={metrics['roc_auc']:.3f}")
        axes[1, 0].plot([0, 1], [0, 1], "--", color="#6b7280", lw=1)
        axes[1, 0].set_title("ROC Curve")
        axes[1, 0].set_xlabel("False Positive Rate")
        axes[1, 0].set_ylabel("True Positive Rate")
        axes[1, 0].legend()

        precision, recall, _ = precision_recall_curve(y_true_bin, y_score)
        axes[1, 1].plot(recall, precision, color="#d97706", lw=2.2)
        axes[1, 1].set_title("Precision-Recall Curve")
        axes[1, 1].set_xlabel("Recall")
        axes[1, 1].set_ylabel("Precision")
        axes[1, 1].set_ylim(0, 1.05)
    else:
        counts = results["y_pred"].value_counts().sort_index()
        axes[1, 0].bar([str(idx) for idx in counts.index], counts.values, color="#0891b2")
        axes[1, 0].set_title("Predicted Class Counts")
        axes[1, 0].set_xlabel("Predicted class")
        axes[1, 0].set_ylabel("Count")
        axes[1, 1].axis("off")
        axes[1, 1].text(
            0.05,
            0.55,
            "No y_score/probability column found.\nROC and precision-recall curves need positive-class scores.",
            fontsize=11,
            va="center",
        )

    fig.tight_layout()
    return fig


def plot_regression_performance(results: pd.DataFrame, metrics: dict[str, Any]) -> plt.Figure:
    plt.style.use("seaborn-v0_8-whitegrid")
    y_true = results["y_true"].astype(float)
    y_pred = results["y_pred"].astype(float)
    residuals = y_pred - y_true

    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    fig.suptitle("Speech Model Regression Performance", fontsize=15)

    axes[0, 0].scatter(y_true, y_pred, alpha=0.78, color="#2563eb", edgecolor="#111827", linewidth=0.35)
    low = float(min(y_true.min(), y_pred.min()))
    high = float(max(y_true.max(), y_pred.max()))
    axes[0, 0].plot([low, high], [low, high], "--", color="#6b7280")
    axes[0, 0].set_title("Predicted vs True")
    axes[0, 0].set_xlabel("True")
    axes[0, 0].set_ylabel("Predicted")

    metric_items = [("MAE", metrics["mae"]), ("RMSE", metrics["rmse"]), ("R2", metrics["r2"])]
    axes[0, 1].bar([name for name, _ in metric_items], [value for _, value in metric_items], color="#2563eb")
    axes[0, 1].set_title("Regression Metrics")

    axes[1, 0].hist(residuals, bins=12, color="#0891b2", alpha=0.82)
    axes[1, 0].axvline(0, color="#111827", lw=1)
    axes[1, 0].set_title("Residual Distribution")
    axes[1, 0].set_xlabel("Predicted - true")
    axes[1, 0].set_ylabel("Count")

    axes[1, 1].scatter(y_pred, residuals, alpha=0.78, color="#d97706", edgecolor="#111827", linewidth=0.35)
    axes[1, 1].axhline(0, color="#111827", lw=1)
    axes[1, 1].set_title("Residuals vs Predicted")
    axes[1, 1].set_xlabel("Predicted")
    axes[1, 1].set_ylabel("Residual")

    fig.tight_layout()
    return fig


def run_labeled_performance_analysis(results_path: str | Path, output_dir: str | Path, task: str = "auto") -> dict[str, Any]:
    results_path = Path(results_path)
    raw_results = pd.read_csv(results_path)
    prepared, task = prepare_model_results(raw_results, task=task)
    if task == "classification":
        metrics = compute_classification_metrics(prepared)
        fig = plot_classification_performance(prepared, metrics)
        plot_path = Path(output_dir) / "model_classification_performance.png"
    else:
        metrics = compute_regression_metrics(prepared)
        fig = plot_regression_performance(prepared, metrics)
        plot_path = Path(output_dir) / "model_regression_performance.png"

    fig.savefig(plot_path, dpi=160, bbox_inches="tight")
    metrics_path = Path(output_dir) / f"model_{task}_metrics.json"
    metrics_path.write_text(json.dumps(_json_ready(metrics), indent=2), encoding="utf-8")
    return {"task": task, "results": prepared, "metrics": metrics, "figure": fig, "plot_path": plot_path, "metrics_path": metrics_path}

In [ ]:
def _prediction_present_mask(series: pd.Series) -> pd.Series:
    text = series.astype("string").str.strip()
    return series.notna() & text.ne("").fillna(False)


def _prediction_column(df: pd.DataFrame) -> str | None:
    return _find_column(
        df,
        ("prediction", "y_pred", "pred", "predicted", "predicted_label", "predicted_mmse", "score_prediction"),
    )


def _infer_prediction_task(path: Path) -> tuple[str, str, str]:
    stem = path.stem.lower().replace("-", "_")
    if "task1" in stem:
        return "classification", "AD classification", "numeric 0/1 diagnosis predictions"
    if "task2" in stem:
        return "regression", "MMSE regression", "numeric MMSE predictions from 0 to 30"
    return "prediction_output", "Prediction output", "non-empty prediction values"


def discover_prediction_output_files(adresso_dir: str | Path, output_dir: str | Path) -> list[Path]:
    discovered: list[Path] = []
    seen: set[Path] = set()

    def append(path: Path) -> None:
        path = path.resolve()
        if path.is_file() and path not in seen:
            discovered.append(path)
            seen.add(path)

    for root in (Path(output_dir), REPO_ROOT / "output_performance" / "speech" / "adresso_submissions", Path(adresso_dir)):
        if not root.exists():
            continue
        for pattern in PREDICTION_OUTPUT_GLOBS:
            for candidate in sorted(root.glob(pattern)):
                append(candidate)

    return discovered


def analyze_prediction_output_file(results_path: str | Path) -> dict[str, Any]:
    results_path = Path(results_path)
    raw_results = pd.read_csv(results_path)
    pred_col = _prediction_column(raw_results)
    if pred_col is None:
        raise ModelPerformanceDataError(f"{results_path} has no Prediction/y_pred column. {PREDICTION_OUTPUT_HINT}")

    id_col = _find_column(raw_results, ("id", "subject_id", "file", "filename"))
    total = int(len(raw_results))
    present_mask = _prediction_present_mask(raw_results[pred_col])
    present_count = int(present_mask.sum())
    task, task_label, validation_rule = _infer_prediction_task(results_path)

    predictions = raw_results.loc[present_mask, pred_col]
    prediction_text = predictions.astype("string").str.strip()
    prediction_numeric = pd.to_numeric(predictions, errors="coerce")

    if task == "classification":
        valid_mask = prediction_numeric.notna() & prediction_numeric.isin([0, 1])
    elif task == "regression":
        valid_mask = prediction_numeric.notna() & prediction_numeric.between(0, 30)
    else:
        valid_mask = pd.Series(True, index=predictions.index)

    valid_count = int(valid_mask.sum()) if present_count else 0
    prediction_counts = {
        str(value): int(count)
        for value, count in prediction_text.value_counts(dropna=True).head(12).items()
    }
    return {
        "source": str(results_path),
        "file": results_path.name,
        "task": task,
        "task_label": task_label,
        "records": total,
        "id_column": str(id_col) if id_col is not None else None,
        "prediction_column": str(pred_col),
        "predictions_filled": present_count,
        "predictions_missing": total - present_count,
        "completion_rate": present_count / total if total else 0.0,
        "numeric_prediction_count": int(prediction_numeric.notna().sum()),
        "valid_prediction_count": valid_count,
        "valid_prediction_rate": valid_count / total if total else 0.0,
        "valid_filled_prediction_rate": valid_count / present_count if present_count else 0.0,
        "validation_rule": validation_rule,
        "prediction_counts": prediction_counts,
        "performance_note": "Accuracy, AUC, MAE, and RMSE require y_true labels.",
    }


def plot_prediction_output_quality(summaries: Sequence[dict[str, Any]]) -> plt.Figure:
    plt.style.use("seaborn-v0_8-whitegrid")
    labels = [Path(summary["file"]).stem.replace("test_results_", "") for summary in summaries]
    x = np.arange(len(labels))

    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    fig.suptitle("Speech Model Output Quality", fontsize=15)

    completion = [summary["completion_rate"] * 100 for summary in summaries]
    valid = [summary["valid_prediction_rate"] * 100 for summary in summaries]
    width = 0.36
    bars_a = axes[0, 0].bar(x - width / 2, completion, width, label="Filled", color="#2563eb")
    bars_b = axes[0, 0].bar(x + width / 2, valid, width, label="Valid", color="#0891b2")
    axes[0, 0].set_title("Prediction Completion")
    axes[0, 0].set_ylabel("Percent of rows")
    axes[0, 0].set_ylim(0, 100)
    axes[0, 0].set_xticks(x)
    axes[0, 0].set_xticklabels(labels, rotation=25, ha="right")
    axes[0, 0].legend()
    for bars in (bars_a, bars_b):
        for bar in bars:
            height = bar.get_height()
            axes[0, 0].text(bar.get_x() + bar.get_width() / 2, min(height + 2, 98), f"{height:.0f}%", ha="center", va="bottom", fontsize=9)

    filled = [summary["predictions_filled"] for summary in summaries]
    missing = [summary["predictions_missing"] for summary in summaries]
    axes[0, 1].bar(x, filled, label="Filled", color="#2563eb")
    axes[0, 1].bar(x, missing, bottom=filled, label="Missing", color="#d97706")
    axes[0, 1].set_title("Rows With Model Output")
    axes[0, 1].set_ylabel("Rows")
    axes[0, 1].set_xticks(x)
    axes[0, 1].set_xticklabels(labels, rotation=25, ha="right")
    axes[0, 1].legend()

    count_items: list[tuple[str, int]] = []
    for summary in summaries:
        file_label = Path(summary["file"]).stem.replace("test_results_", "")
        for value, count in summary["prediction_counts"].items():
            count_items.append((f"{file_label}: {value}", count))
    if count_items:
        count_items = sorted(count_items, key=lambda item: item[1], reverse=True)[:12]
        axes[1, 0].barh([item[0] for item in count_items], [item[1] for item in count_items], color="#2563eb")
        axes[1, 0].invert_yaxis()
        axes[1, 0].set_title("Prediction Value Counts")
        axes[1, 0].set_xlabel("Rows")
    else:
        axes[1, 0].axis("off")
        axes[1, 0].text(0.05, 0.55, "No predictions are filled in the discovered model-output files.", fontsize=11, va="center")

    axes[1, 1].axis("off")
    status_lines = []
    for summary in summaries:
        status_lines.append(f"{summary['file']}: {summary['predictions_filled']}/{summary['records']} filled, {summary['valid_prediction_count']} valid")
        status_lines.append(f"Rule: {summary['validation_rule']}")
    status_lines.append("")
    status_lines.append("Labeled y_true values are required for accuracy, AUC, MAE, or RMSE.")
    axes[1, 1].text(0.0, 1.0, "\n".join(status_lines), fontsize=10, va="top")

    fig.tight_layout()
    return fig


def run_prediction_output_analysis(results_paths: Sequence[str | Path], output_dir: str | Path) -> dict[str, Any]:
    summaries = [analyze_prediction_output_file(path) for path in results_paths]
    if not summaries:
        raise ModelPerformanceDataError(f"No usable prediction-output files found. {PREDICTION_OUTPUT_HINT}")

    plot_path = Path(output_dir) / "model_output_quality.png"
    metrics_path = Path(output_dir) / "model_output_quality_metrics.json"
    total_filled = sum(summary["predictions_filled"] for summary in summaries)
    payload = {
        "kind": "prediction_output_quality",
        "summaries": summaries,
        "performance_note": "These metrics come from model-output files; labeled y_true values are required for true performance scores.",
    }
    if total_filled == 0:
        plot_path.unlink(missing_ok=True)
        payload["performance_note"] = (
            "Discovered prediction-output files are empty; no model-output figure was generated. "
            "Fill Prediction values or provide labeled y_true/y_pred results."
        )
        metrics_path.write_text(json.dumps(_json_ready(payload), indent=2), encoding="utf-8")
        return {"kind": "prediction_output_quality", "summaries": summaries, "figure": None, "plot_path": None, "metrics_path": metrics_path}

    fig = plot_prediction_output_quality(summaries)
    fig.savefig(plot_path, dpi=160, bbox_inches="tight")
    metrics_path.write_text(json.dumps(_json_ready(payload), indent=2), encoding="utf-8")
    return {"kind": "prediction_output_quality", "summaries": summaries, "figure": fig, "plot_path": plot_path, "metrics_path": metrics_path}

In [ ]:
results = run_analysis()
summary = results["summary"]
MODEL_RESULTS_CSV = results["performance_results_hint"]
PREDICTION_OUTPUT_FILES = discover_prediction_output_files(results["adresso_dir"], results["output_dir"])

print(f"Feature table written: {results['stats_path']}")
print(f"Sessions: {summary['sessions']}")
print(f"Segments: {summary['segments']}")
print(f"Performance input expected at: {MODEL_RESULTS_CSV}")
print(f"Prediction output files found: {len(PREDICTION_OUTPUT_FILES)}")
for path in PREDICTION_OUTPUT_FILES:
    print(f"Prediction output file: {path}")

In [ ]:
performance = None
performance_metrics = None
output_quality = None
output_quality_metrics = None

if MODEL_RESULTS_CSV.exists():
    try:
        performance = run_labeled_performance_analysis(MODEL_RESULTS_CSV, results["output_dir"])
        performance_metrics = performance["metrics"]
        print(f"Task: {performance['task']}")
        print(f"Metrics JSON: {performance['metrics_path']}")
        print(f"Performance figure: {performance['plot_path']}")
    except ModelPerformanceDataError as exc:
        print(f"No labeled model-performance figure generated: {exc}")

if performance is None and PREDICTION_OUTPUT_FILES:
    output_quality = run_prediction_output_analysis(PREDICTION_OUTPUT_FILES, results["output_dir"])
    output_quality_metrics = output_quality["summaries"]
    print(f"Model output quality metrics: {output_quality['metrics_path']}")
    if output_quality["plot_path"] is not None:
        print(f"Model output quality figure: {output_quality['plot_path']}")
    else:
        print("No model-output figure generated: discovered prediction files are empty.")
        print("For actual performance, provide model_performance_results.csv with y_true and y_pred columns.")
elif performance is None:
    print("No labeled model results or prediction-output files found.")
    print(PERFORMANCE_COLUMN_HINT)
    print(PREDICTION_OUTPUT_HINT)

performance["figure"] if performance is not None else output_quality["figure"] if output_quality is not None and output_quality["figure"] is not None else None

In [ ]:
performance_metrics or output_quality_metrics